In [0]:
# %pip install openpyxl

In [0]:
%restart_python

In [0]:
import openpyxl
print(openpyxl.__version__)

In [0]:
from openpyxl import load_workbook

file_path="/Volumes/main/training/hr_data/HR_Dataset.xlsx"

workbook=load_workbook(file_path,data_only=True)
sheet=workbook["employees"]
data=list(sheet.values)
headers=data[0]
rows=data[1:]

print(headers)
print("Number of rows:",len(rows))

employees=spark.createDataFrame(rows,schema=headers)
display(employees)

Question 1
Task Requirements:
Compute the highest and lowest individual base salaries across all active personnel.
Determine the overall average base salary across the entire workforce alongside total monthly payroll obligation.
Calculate the maximum and average incentive rate strictly among staff eligible for commissions.
Ensure all currency and rate figures are rounded to two decimal places for financial reporting.
Expected Output Specification: DataFrame with a single summary row containing [max_salary, min_salary, avg_salary, total_salary, max_commission, avg_commission].

In [0]:
from pyspark.sql.functions import sum,avg,min,max,when,col,round,cast

salary_summary=employees.agg(
    sum("salary") .cast("decimal(10,2)").alias("total_salary"),
    max("salary").cast("decimal(10,2)").alias("max_salary"),
    min("salary").cast("decimal(10,2)").alias("min_salary"),
    avg("salary").cast("decimal(10,2)").alias("avg_salary"),
    round(
        max(
            when(
                col("commission_pct").isNotNull(),
                col("commission_pct")
            )
        ),
        2
    ).alias("max_commission"),
    round(
        avg(
            when(
                col("commission_pct").isNotNull(),
                col("commission_pct")
            )
        ),
        2
    ).alias("avg_commission")
)

display(salary_summary)

Question 2
Task Requirements:
•	Aggregate total staff headcount allocated to each distinct department.
•	Ensure employees without an assigned department are not discarded and appear under an unassigned classification.
•	Arrange the results in ascending order of department identifier.
Expected Output Specification: DataFrame with columns [department_id, employee_count], ordered by department_id.


In [0]:
from pyspark.sql.functions import col,when,count

result=employees.withColumn(
    "department_id",
    when(col("department_id").isNull(),"Unassigned")
    .otherwise(col("department_id").cast("string"))
).groupBy("department_id").agg(
    count("employee_id").alias("employee_count")
).orderBy(col("department_id")
)


display(result)

Question 3
Task Requirements:
•	Group workforce records by department.
•	Compute the cumulative monthly salary expenditure and the average employee compensation for each department.
•	Round departmental average salaries to two decimal places.
•	Order the final breakdown in descending order of total payroll spend.
Expected Output Specification: DataFrame with columns [department_id, total_salary_payout, avg_department_salary], ordered highest spend first.


In [0]:
from pyspark.sql.functions import col,sum,avg,round

result=employees.groupBy("department_id").agg(
    sum("salary").alias("total_salary_payout"),
    round(avg("salary"),2).alias("avg_department_salary")
).orderBy(col("total_salary_payout").desc())


display(result)

Question 4 Task Requirements:
•	Calculate the total count of staff who participate in incentive compensation (commission recorded).
•	Calculate the total count of staff on fixed compensation (commission missing).
•	Determine what percentage of the total corporate headcount receives commission incentives.
Expected Output Specification: DataFrame with columns [commission_earners_count, non_commission_count, commission_earners_pct].



In [0]:
from pyspark.sql.functions import col,count

result=employees.agg(
    count(
        when(
            col("commission_pct").isNull(),1))
                .alias("non_commission_count"),
    count(
        when(
            col("commission_pct").isNotNull(),1))
                .alias("commission_earners_count"),
    round(
        count(
            when(col("commission_pct").isNotNull(),1))
        *100.0/count("employee_id"),2)
            .alias("commission_earners_pct")
)

display(result)

Question 5 Task Requirements:
•	Group personnel by job classification.
•	Determine the minimum and maximum actual salary paid within each job code.
•	Calculate the absolute salary spread (difference between top earner and bottom earner) in each role.
•	Restrict the analysis to job roles currently staffed by more than one employee.
•	Sort the findings from widest wage spread to narrowest.
Expected Output Specification: DataFrame with columns [job_id, min_actual_salary, max_actual_salary, salary_spread], ordered descending by spread.


In [0]:
from pyspark.sql.functions import min,max,when,col

result=employees.groupBy("job_id").agg(
    count("employee_id").alias("employee_count"),
    min("salary").alias("min_actual_salary"),
    max("salary").alias("max_actual_salary")
).filter(
    col("employee_count")>1
).withColumn(
    "salary_spread",
    col("max_actual_salary")-col("min_actual_salary")
).drop("employee_count").orderBy(
    col("salary_spread").desc()
)

display(result)

Question 6 
Input Tables / Columns: employees (department_id, job_id)
Task Requirements:
•	Aggregate workforce data by department.
•	Calculate the count of distinct job roles active within each department.
•	Include total departmental headcount alongside the distinct job count.
•	Sort departments starting with the highest variety of distinct roles.
Expected Output Specification: DataFrame with columns [department_id, distinct_job_count, total_headcount], ordered descending by distinct jobs.

In [0]:
from pyspark.sql.functions import col,countDistinct,count

result=employees.groupBy("department_id").agg(
    countDistinct("job_id").alias("distinct_job_count"),
    count("employee_id").alias("total_headcount")
    
).orderBy(col("distinct_job_count").desc())

display(result)



In [0]:
from openpyxl import load_workbook

file_path="/Volumes/main/training/hr_data/HR_Dataset.xlsx"

workbook=load_workbook(file_path,data_only=True)
sheet=workbook["departments"]
data=list(sheet.values)
headers=data[0]
rows=data[1:]

print(headers)
print("Number of rows:",len(rows))

departments=spark.createDataFrame(rows,schema=headers)
display(departments)

Question 7 
Task Requirements:
•	Associate each employee with their facility location via department master records.
•	Group the combined data by location identifier.
•	Compute the total headcount and total payroll expenditure assigned to each physical location.
•	Order the output from most populated facility to least populated.
Expected Output Specification: DataFrame with columns [location_id, total_headcount, total_location_payroll], sorted by headcount descending.


In [0]:
from pyspark.sql.functions import col

result=employees.join(
    departments,
    employees["department_id"]==departments["department_id"],
    "inner"
).groupBy(
    departments["location_id"]
).agg(
    count(employees["employee_id"]).alias("total_headcount"),
    sum(employees["salary"]).alias("total_location_payroll")

).orderBy(
    col("total_headcount").desc()
).select(
    departments["location_id"].alias("location_id"),
    col("total_headcount"),
    col("total_location_payroll")
)

display(result)



In [0]:
# from pyspark.sql.functions import col,when,sum,avg,round


# result = employees.join(
#     departments,
#     "department_id",
#     "inner"
# ).groupBy(
#     "location_id"
# ).agg(
#     count("employee_id").alias("total_headcount"),
#     sum("salary").alias("Total_salary")
# ).orderBy(
#     col("headcount").desc()
# ).select(
#     col("location_id"),
#     col("total_headcount"),
#     col("Total_salary")
# )
# display(result)

Question 8
Task Requirements:
•	Group employees simultaneously by department and job code.
•	Compute headcount and total salary payout for each specific job within each department.
•	Display average compensation per role rounded to two decimal places.
•	Order results by department identifier ascending, then by total payroll descending within each department.
Expected Output Specification: DataFrame with columns [department_id, job_id, headcount, total_payroll, avg_salary].


In [0]:
from pyspark.sql.functions import col,when,count,avg,round,sum

result=employees.groupBy(
    "department_id","job_id"
).agg(
    count("employee_id").alias("headcount"),
    sum("salary").alias("total_payroll"),
    round(avg("salary"),2).alias("avg_salary")
).orderBy(
    col("department_id"),
    col("total_payroll").desc()
)

display(result)

Question 9
Task Requirements:
•	Compute total headcount and aggregate payroll per department.
•	Isolate departments that satisfy both business criteria: total headcount of 5 or more employees, and total monthly payroll exceeding $30,000.
•	Sort qualified departments from highest spending to lowest spending.
Expected Output Specification: Filtered DataFrame with columns [department_id, headcount, total_payroll].


In [0]:
from pyspark.sql.functions import col,when,count,sum


result=employees.groupBy(
    "department_id"
).agg(
    count("employee_id").alias("headcount"),
    sum("salary").alias("total_payroll")
).filter(
    (col("headcount") >= 5) & 
    (col("total_payroll")>30000)
).orderBy(
    col("total_payroll").desc()
)

display(result)

Question 10 
Task Requirements:
•	Group employees by manager identifier, excluding records where the manager identifier is missing.
•	Calculate total direct reports supervised by each manager.
•	Calculate total monthly payroll managed by each manager across their direct subordinates.
•	Sort managers from largest direct team to smallest direct team.
Expected Output Specification: DataFrame with columns [manager_id, direct_reports_count, total_supervised_payroll].


In [0]:
from pyspark.sql.functions import col

result=employees.filter(col("manager_id").isNotNull()).groupBy(
    "manager_id"
).agg(
    count("employee_id").alias("direct_reports_count"),
    sum("salary").alias("total_supervised_payroll")
).orderBy(
    col("direct_reports_count").desc()
)

display(result)


Task Requirements:
•	Extract the calendar year from each employee's official hire date.
•	Group workforce data by hiring year.
•	Compute total recruits hired, lowest starting wage, highest starting wage, and average compensation per hiring class.
•	Order chronologically by hire year.
Expected Output Specification: DataFrame with columns [hire_year, total_hires, min_salary, max_salary, avg_salary], sorted by hire_year ASC.


In [0]:
from pyspark.sql.functions import col,year,sum,cast,avg,count

result=employees.withColumn(
    "hire_year",
    year("hire_date")
).groupBy(
    "hire_year"
).agg(
    count("employee_id").alias("total_hires"),
    min("salary").alias("min_salary"),
    max("salary").alias("max_salary"),
   avg("salary").cast("decimal(10,2)").alias("avg_salary")
).orderBy(
    col("hire_year")
)

display(result)

Question 12 Task Requirements:
•	Group personnel by department.
•	Count staff earning $10,000 or more per month (Executive Tier).
•	Count staff earning between $5,000 and $9,999 per month (Mid Tier).
•	Count staff earning under $5,000 per month (Associate Tier).
•	Include total departmental headcount and verify tier counts sum to total headcount.
Expected Output Specification: DataFrame with columns [department_id, total_staff, exec_count, mid_count, associate_count].


In [0]:
from pyspark.sql.functions import count,col,when,sum
result=employees.filter(col("department_id").isNotNull()).groupBy(
    "department_id"
).agg(
    count("salary").alias("total_staff"),
    count(
        when(
            col("salary")>=10000,1
            )
        ).alias("Executive_Tier"),
    count(
        when(
            (col("salary")>=5000) & (col("salary")<=9999),1
        )
    ).alias("Mid_Tier"),
    count(
        when(
            col("salary")<5000,1
        )
    ).alias("Associate_Tier")
).orderBy("department_id")

display(result)

Question 13 Task Requirements:
•	Calculate actual dollar commission payout for each eligible employee based on their base salary and commission rate.
•	Aggregate by department to calculate: total base salary spend, total dollar commission spend, and combined gross compensation.
•	Compute the effective incentive percentage: total commission dollars divided by total base salary dollars.
•	Filter for departments that have non-zero commission expenses.
Expected Output Specification: DataFrame with columns [department_id, total_base_payroll, total_commission_spend, gross_payroll, effective_commission_rate].


In [0]:
from pyspark.sql.functions import col, sum

employees = employees.withColumn(
    "commission_dollars",
    col("salary") * col("commission_pct")
)

result = employees.groupBy("department_id").agg(
    sum("salary").alias("total_base_payroll"),
    sum("commission_dollars").alias("total_commission_spend")
)

result = result.withColumn(
    "gross_payroll",
    col("total_base_payroll") + col("total_commission_spend")
)

result = result.withColumn(
    "effective_commission_rate",
    col("total_commission_spend") / col("total_base_payroll")
)

result = result.filter(
    col("total_commission_spend") > 0
)

result = result.select(
    "department_id",
    "total_base_payroll",
    "total_commission_spend",
    "gross_payroll",
    "effective_commission_rate"
)

result.show()

Task Requirements:
•	Transform the employee table so each unique department forms a row.
•	Rotate job classifications into distinct horizontal column headers.
•	Populate matrix cells with total headcount for each department-job intersection.
•	Ensure departments with zero staff in a specific role display 0 instead of missing values.
Expected Output Specification: Pivoted DataFrame where rows are department_id and column headers are distinct job codes.


In [0]:
from pyspark.sql.functions import when,col


result=employees.groupBy("department_id").pivot("job_id").count().fillna(0)


result.show()

Question 15 Task Requirements:
•	Group staff by department, filtering to departments with at least 3 employees.
•	Compute standard statistical metrics: minimum, maximum, mean, sample standard deviation, and variance of salaries.
•	Calculate the 25th, 50th (median), and 75th approximate salary percentiles.
•	Calculate the Interquartile Range (IQR) for each department.
Expected Output Specification: DataFrame with columns [department_id, min_sal, max_sal, mean_sal, stddev_sal, p25_sal, median_sal, p75_sal, iqr].


In [0]:
from pyspark.sql.functions import col,stddev_samp,percentile_approx


result=employees.groupBy("department_id").agg(
    count("department_id").alias("count_department_id"),
    min("salary").alias("min_sal"),
    max("salary").alias("max_salary"),
    round(avg("salary"),2).alias("mean_salary"),
    round(stddev_samp("salary"),2).alias("stddev_sal"),
    round(percentile_approx("salary",0.25),2).alias("p25_sal"),
    round(percentile_approx("salary",0.50),2).alias("median_sal"),
    round(percentile_approx("salary",0.75),2).alias("p75_sal")
).filter(col("count_department_id")>3)

result=result.withColumn(
    "IQR",
    col("p75_sal")-col("p25_sal")
)

result=result.select(
    "department_id",
    "min_sal",
    "max_salary",
    "mean_salary",
    "stddev_sal",
    "p25_sal",
    "median_sal",
    "p75_sal",
    "IQR"
)


display(result)

Question 16
Task Requirements:
•	Combine employee compensation data with department location details.
•	Format the dataset so department names form vertical rows and location identifiers form horizontal columns.
•	Aggregate total salary spend inside each intersection cell.
•	Replace missing intersections with 0.00 and order by department name.
Expected Output Specification: Pivoted DataFrame with department_name as rows and location identifiers as column headers.


In [0]:
from pyspark.sql.functions import col,sum


result=employees.join(
    departments,
    employees["department_id"]==departments["department_id"],
    "inner"
).groupBy("department_name").pivot("location_id").agg(
    sum("salary")
).fillna(0).orderBy(col("department_name"))

display(result)


Question 17 Input Tables / Columns: employees (department_id, salary)
Task Requirements:
•	Calculate the overall corporate average salary as a baseline scalar metric across all employees.
•	Calculate the average salary within each department.
•	Compute the absolute dollar variance: departmental average minus company-wide average.
•	Calculate the percentage deviation of each department from the company benchmark.
•	Tag each department as 'ABOVE BENCHMARK' or 'BELOW BENCHMARK' based on sign of deviation.
Expected Output Specification: DataFrame with columns [department_id, dept_avg_salary, company_avg_salary, dollar_variance, pct_deviation, benchmark_status].


In [0]:
from pyspark.sql.functions import col,avg,lit,round

#company average

overall_avg=employees.agg(
    avg("salary").alias("overall_average")
).collect()[0]["overall_average"]


#department average
result=employees.groupBy("department_id").agg(
    round(avg("salary"),2).alias("dept_avg_salary")
)

#company average
result=result.withColumn(
    "company_avg_salary",
    lit(overall_avg)
)

#dollar_variance
result=result.withColumn(
    "dollar_variance",
    round(col("dept_avg_salary")-lit(overall_avg),2)
)

#percentage deviation
result=result.withColumn(
    "pct_deviation",
    (
       (col("dept_avg_salary"))-(lit(overall_avg))/lit(overall_avg))
)

#bench mark

result=result.withColumn(
    "benchmark_status",
    when(
        col("pct_deviation")>0,
        "Above benchmark"
    ).otherwise(
        "below benchmark"
    )
)

display(result)

Question 18 Task Requirements:
•	Aggregate by department for departments having at least 2 staff members.
•	Identify the highest salary and lowest salary paid in each department.
•	Calculate the Polarization Ratio: highest salary divided by lowest salary.
•	Flag departments where the top earner makes more than 2.5 times the lowest earner as 'HIGH SPREAD'.
•	Order by Polarization Ratio descending.
Expected Output Specification: DataFrame with columns [department_id, min_sal, max_sal, polarization_ratio, spread_flag].


In [0]:
from pyspark.sql.functions import col,when,min,max,round,when

result=employees.groupBy("department_id").agg(
    count("employee_id").alias("staff_count"),
    min("salary").alias("min_sal"),
    max("salary").alias("max_sal")
).filter(
    col("staff_count")>=2
)

result=result.withColumn(
    "polarization_ratio",
    round(
        col("max_sal")/col("min_sal"),2
    )
)


result=result.withColumn(
    "spread_flag",
    when(
        (col("max_sal"))>(2.5
        * col("min_sal")),"HIGH SPREAD"
    ).otherwise(
        "NORMAL"
    )
)

result=result.orderBy(
    col("polarization_ratio").desc()
)

result=result.select(
    "department_id",
    "min_sal",
    "max_sal",
    "polarization_ratio",
    "spread_flag"
)


display(result)


Question 19 Task Requirements:
•	Identify all employees who serve as managers by matching employee records with subordinate supervisor identifiers.
•	Calculate the cumulative subordinate payroll and total headcount supervised by each manager.
•	Rank managers based on their personal compensation and isolate the top 3 highest-earning managers.
•	Display manager personal salary alongside the total subordinate payroll they oversee.
Expected Output Specification: DataFrame with 3 rows containing [manager_id, manager_personal_salary, subordinate_count, total_subordinate_payroll].

In [0]:
from pyspark.sql.functions import when,col,round,count

e=employees.alias("e")
m=employees.alias("m")

result=e.join(
    m,
    e["manager_id"]==m["employee_id"],
    "inner"
)
#employee count
result=result.groupBy(
    col("m.employee_id").alias("manager_id")
    ).agg(
    count("e.employee_id").alias("subordinate_count"),
    sum(col("e.salary")).alias("total_subordinate_payroll")
)
    
manager_salary=employees.select(
    col("employee_id").alias("manager_id"),
    col("salary").alias("manager_personal_salary")
)

result=result.join(
    manager_salary,
    "manager_id",
    "inner"
)

result=result.orderBy(
    col("manager_personal_salary").desc()
).limit(3)

display(result)

Question 20
Input Tables / Columns: employees (department_id, job_id, salary)
Task Requirements:
•	Calculate total payroll expense and total headcount grouped hierarchically by department and job role.
•	In the same result set, include subtotal summaries for each department across all its jobs.
•	Include an overall grand total summary row representing the entire company spend.
•	Order the output so that department subtotals and grand totals appear in a logical sequence.
Expected Output Specification: Hierarchical DataFrame containing granular job rows, department subtotals, and the company grand total.


In [0]:
from pyspark.sql.functions import col,when

# 1.department + job level
detail=employees.groupBy(
    col("department_id"),
    col("job_id")
).agg(
    sum("salary").alias("total_payroll"),
    count("employee_id").alias("headcount")
)

# 2.department subtotal
department_total=employees.groupBy(
    col("department_id")
).agg(
    sum("salary").alias("total_payroll"),
    count("employee_id").alias("headcount")
).withColumn(
    "job_id",
    lit(None)
)

# 3.Grand total
grand_total= employees.agg(
    sum("salary").alias("total_payroll"),
    count("employee_id").alias("headcount")
).withColumn(
    "job_id",
    lit(None)
).withColumn(
    "department_id",
    lit(None)
)

# 4. union all

result=detail.unionByName(
    department_total
).unionByName(
    grand_total
).orderBy(col("department_id"),col("job_id"))


display(result)

In [0]:
from pyspark.sql.functions import col, sum, count

result = employees.rollup(
    col("department_id"),
    col("job_id")
).agg(
    sum("salary").alias("total_payroll"),
    count("employee_id").alias("headcount")
).orderBy(col("department_id"),col("job_id"))
display(result)

Question 21
Task Requirements:
•	Create a boolean indicator indicating whether each employee receives commission incentives.
•	Generate aggregations across every possible combination subset of department, job, and commission eligibility.
•	For every combination subset, compute total payroll and employee count.
•	Explain why multi-dimensional grouping combinations produce 2^N aggregation levels compared to hierarchical N+1 levels.
Expected Output Specification: Multi-dimensional Cube DataFrame containing all combination permutations of the grouping columns.


cube ->3 dimension
1. Department + Job + Commission
2. Department + Job
3. Department + Commission
4. Job + Commission
5. Department
6. Job
7. Commission
8. Grand Total

In [0]:
from pyspark.sql.functions import col,when,sum,count

employee_with_commission=employees.withColumn(
    "commission_eligible",
    when(
        col("commission_pct").isNotNull(),True)
    .otherwise(
        False
    )
)

result=employee_with_commission.cube(
    "department_id",
    "job_id",
    "commission_eligible"
).agg(
    sum("salary").alias("total_payroll"),
    count("*").alias("employee_count")
)
display(result)


Question 22. Input Tables / Columns: employees (department_id, job_id, salary)
Task Requirements:
•	Perform a hierarchical multi-level aggregation across department and job code, calculating total salary.
•	Extract the grouping bitmask integer that uniquely identifies the level of aggregation for each row.
•	Decode the bitmask into human-readable labels: 'Detail Record', 'Department Subtotal', and 'Company Grand Total'.
•	Ensure actual null values in raw data are not confused with generated subtotal summaries.
Expected Output Specification: DataFrame with explicit aggregation level labels distinguishing granular records from summary subtotals.
Guiding Hints & Formulas: Formula / Concept: PySpark provides grouping metadata functions (such as grouping_id) that return bitmasks indicating which dimensions are aggregated in each row.


In [0]:
from pyspark.sql.functions import col,sum,col,when,grouping_id


result=employees.rollup("department_id","job_id"
).agg(
    sum(col("salary")).alias("total_salary"),
    grouping_id().alias("grouping_id")
).withColumn(
    "aggregation_level",
    when(
        col("grouping_id")==0 ,"Detail Record"
    ).when(
        col("grouping_id")==1 , "Department Subtotal"
    ).when(
        col("grouping_id")==3, "Company Grand Total"
    )
)

display(result)


Question 23. Input Tables / Columns: employees (department_id, job_id, salary)
Task Requirements:
•	Perform a hierarchical multi-level aggregation across department and job code, calculating total salary.
•	Extract the grouping bitmask integer that uniquely identifies the level of aggregation for each row.
•	Decode the bitmask into human-readable labels: 'Detail Record', 'Department Subtotal', and 'Company Grand Total'.
•	Ensure actual null values in raw data are not confused with generated subtotal summaries.
Expected Output Specification: DataFrame with explicit aggregation level labels distinguishing granular records from summary subtotals.
Guiding Hints & Formulas: Formula / Concept: PySpark provides grouping metadata functions (such as grouping_id) that return bitmasks indicating which dimensions are aggregated in each row.


In [0]:
from pyspark.sql.functions import col,when,lit

#step 1 -> join the employees table and department table
join_df=employees.join(
    departments,
    employees["department_id"]==departments["department_id"],
    "inner"
)

#step 2 -> Department-level aggregation
dept_df=join_df.groupBy(
    employees.department_id,
    departments.department_name
).agg(
    count("employee_id").alias("headcount"),
    sum("salary").alias("base_payroll"),
    sum("commission_dollars").alias("commission_payout")
)

#step 3 -> calculate total department spend and average employee cost

dept_df=dept_df.withColumn(
    "total_dept_spend",
    col("base_payroll")+col("commission_payout")
).withColumn(
    "avg_employee_cost",
    col("total_dept_spend")/col("headcount")
)

#step 4 -> calculate enterprise total cross spend
enterprise_total= dept_df.agg(
    sum("total_dept_spend").alias("enterprise_total_spend")
).collect()[0]["enterprise_total_spend"]

#step 5 -> Calculate budget share %
dept_df = dept_df.withColumn(
    "budget_share_pct",
    (col("total_dept_spend")/lit(enterprise_total) )* 100
)

#step 6 -> Assign governance status
dept_df= dept_df.withColumn(
    "governance_flag",
    when(col("budget_share_pct")>=25,"CRITICAL SPEND")
    .when(col("budget_share_pct")>=10, "HIGH SPEND")
    .otherwise("NORMAL SPEND")
)

#step 7 -> select required columns and order by budget share descending

final_df= dept_df.select(
    "department_id",
    "department_name",
    "headcount",
    "base_payroll",
    "commission_payout",
    "total_dept_spend",
    "avg_employee_cost",
    "budget_share_pct",
    "governance_flag"
).orderBy(
    col("budget_share_pct").desc()
)

final_df.show()

Question 24. Input Tables / Columns: employees (department_id, job_id, salary)
Task Requirements:
•	Filter workforce data to departments maintaining at least 4 active staff members.
•	For each department compute: minimum, maximum, average, median salary, and sample standard deviation.
•	Calculate the Wage Disparity Ratio: highest salary divided by lowest salary.
•	Calculate the Coefficient of Variation (CV) percentage: standard deviation divided by average salary multiplied by 100.
•	Categorize Equity Risk: 'HIGH DISPARITY' (ratio > 3.0 or CV > 50%), 'MODERATE' (ratio between 2.0 and 3.0), or 'EQUITABLE' (ratio < 2.0).
•	Order departments by Wage Disparity Ratio descending.
Expected Output Specification: DataFrame with columns [department_id, headcount, min_salary, max_salary, avg_salary, median_salary, salary_stddev, disparity_ratio, coeff_variation_pct, equity_risk_level].
Guiding Hints & Formulas: Formula / Concept: Coefficient of Variation = (StdDev / Mean) * 100. Disparity Ratio = Max / Min. Utilize approximate percentiles and sample dispersion functions.



In [0]:
from pyspark.sql.functions import sum,col,when,avg,max,min,count,percentile_approx,stddev_samp,round


result=employees.groupBy(
    col("department_id")
).agg(
    count(col("employee_id")).alias("headcount"),
    min(col("salary")).alias("lowest_salary"),
    max(col("salary")).alias("highest_salary"),
    round(avg(col("salary")),2).alias("avg_salary"),
    round(percentile_approx(col("salary"),0.5),2).alias("median_salary"),
    round(stddev_samp(col("salary")),2).alias("salary_stddev")
)
# Filter workforce data to departments maintaining at least 4 active staff members. 
result=result.filter(col("headcount")>=4)

# Calculate the Wage Disparity Ratio: highest salary divided by lowest salary
result=result.withColumn(
    "disparity_ratio",
    round(col("highest_salary")/col("lowest_salary"),2)
)

#Calculate the Coefficient of Variation (CV) percentage: standard deviation divided by average salary multiplied by 100.
result=result.withColumn(
    "coeff_variation_pct",
    round(col("salary_stddev")/col("avg_salary")*100,2)
)

#Categorize Equity Risk: 'HIGH DISPARITY' (ratio > 3.0 or CV > 50%), 'MODERATE' (ratio between 2.0 and 3.0), or 'EQUITABLE' (ratio < 2.0)
result=result.withColumn(
    "equity_risk_level",
    when(
        (col("disparity_ratio") > 3.0) |
        (col("coeff_variation_pct") > 50),
        "HIGH_DISPARITY"
    )
    .when(
        (col("disparity_ratio") >= 2.0) &
        (col("disparity_ratio")<=3.0),
        "MODERATE"
    )
    .when(
        (col("disparity_ratio")<2.0),
        "EQUITABLE"
    )
)

result=result.orderBy(col("disparity_ratio").desc())


display(result)

Question 25. Input Tables / Columns: employees (department_id=80), sales performance data
Task Requirements:
•	Calculate base commission payout: base salary multiplied by commission rate (defaulting to 5% if rate is missing).
•	Apply Performance Tier Bonus: if commission rate is 30% or higher, add $2,500; if between 15% and 29%, add $1,200; otherwise $0.
•	Apply Tenure Loyalty Multiplier: if employee was hired before January 1, 2005, award an additional 5% bonus on base salary.
•	Reconcile at employee level: Total Gross Payout = Base Salary + Base Commission + Tier Bonus + Tenure Bonus.
•	Compute department summary: Total Gross Payout, Total Incentive Burden, and Effective Incentive Rate across the sales force.
•	Verify that zero final payouts contain null or negative amounts.
Expected Output Specification: Reconciliation DataFrame with columns [employee_id, full_name, base_salary, base_commission, quota_bonus, seniority_bonus, total_gross_payout, effective_incentive_rate_pct].
Guiding Hints & Formulas: Formula / Concept: Total Payout = Base + Commission + Quota Bonus + Seniority Bonus. Effective Rate = ((Total Payout - Base) / Base) * 100. Ensure complete null safety.


In [0]:
from pyspark.sql import functions as F

# 1. Filter Sales Department
sales_employees = employees.filter(
    F.col("department_id") == 80
)

# 2. Handle Missing Commission Rate
# If commission_pct is NULL, use 5%
sales_employees = sales_employees.withColumn(
    "effective_commission_rate",
    F.when(
        F.col("commission_pct").isNull(),
        0.05
    ).otherwise(
        F.col("commission_pct")
    )
)

# 3. Calculate Base Commission
sales_employees = sales_employees.withColumn(
    "base_commission",
    F.col("salary") * F.col("effective_commission_rate")
)

# 4. Calculate Performance Tier Bonus
sales_employees = sales_employees.withColumn(
    "quota_bonus",
    F.when(
        F.col("effective_commission_rate") >= 0.30,
        2500
    ).when(
        F.col("effective_commission_rate") >= 0.15,
        1200
    ).otherwise(
        0
    )
)

# 5. Calculate Tenure / Seniority Bonus
sales_employees = sales_employees.withColumn(
    "seniority_bonus",
    F.when(
        F.col("hire_date") < "2005-01-01",
        F.col("salary") * 0.05
    ).otherwise(
        0
    )
)

# 6. Calculate Total Gross Payout
sales_employees = sales_employees.withColumn(
    "total_gross_payout",
    F.col("salary")
    + F.col("base_commission")
    + F.col("quota_bonus")
    + F.col("seniority_bonus")
)

# 7. Calculate Effective Incentive Rate
sales_employees = sales_employees.withColumn(
    "effective_incentive_rate_pct",
    F.when(
        F.col("salary") > 0,
        (
            (F.col("total_gross_payout") - F.col("salary"))
            / F.col("salary")
        ) * 100
    ).otherwise(
        0
    )
)

# 8. Employee-level Reconciliation
result = sales_employees.select(
    "employee_id",
    F.concat_ws(
        " ",
        F.col("first_name"),
        F.col("last_name")
    ).alias("full_name"),
    F.col("salary").alias("base_salary"),
    "base_commission",
    "quota_bonus",
    "seniority_bonus",
    "total_gross_payout",
    "effective_incentive_rate_pct"
)

# Round values
result = result.withColumn(
    "base_commission",
    F.round(F.col("base_commission"), 2)
).withColumn(
    "quota_bonus",
    F.round(F.col("quota_bonus"), 2)
).withColumn(
    "seniority_bonus",
    F.round(F.col("seniority_bonus"), 2)
).withColumn(
    "total_gross_payout",
    F.round(F.col("total_gross_payout"), 2)
).withColumn(
    "effective_incentive_rate_pct",
    F.round(F.col("effective_incentive_rate_pct"), 2)
)

display(result)

Question 26. Input Tables / Columns: wide_quarterly_sales (department_id, Q1_Revenue, Q2_Revenue, Q3_Revenue, Q4_Revenue)
Task Requirements:
•	Examine the wide DataFrame where financial quarters are stored horizontally as distinct column attributes.
•	Transform the horizontal quarterly columns into normalized vertical rows such that each record represents a single department and quarter.
•	Retain three normalized columns: department identifier, quarter label ('Q1', 'Q2', etc.), and revenue amount.
•	Verify that the sum of quarterly revenues in the normalized dataset matches total annual revenue from the wide table exactly.
•	Sort the normalized dataset by department identifier and quarter.
Expected Output Specification: Normalized DataFrame with 3 columns: [department_id, Quarter, Revenue].
Guiding Hints & Formulas: Formula / Concept: Relational unpivoting transforms M columns into M rows per key. Investigate PySpark SQL stack expressions or array explode patterns to normalize wide tables.
